# So far:

- prime generation with seed

- rough sizes of numbers known

- totient function, modular inverse implemented

- easy-ish imports of functions

- final decryption takes a very long time, this is maybe a bug I will need to fix manually

# To Do:

- Plaintext to ASCII and back 

   - Probably define a class for this

- Either fix decryption bug or change size of primes.

- Figure out distribution

In [1]:
import project_path
from rsatools import carmichael, inverse257, prime_rng, PrimeGenerator
import codecs
from itertools import chain


So now I can just import from the `rsatools` package I defined.  That's quite nice and should be good for playing around and starting new notebooks.  

So what's the priority right now? I think the first thing to do is getting ASCII conversion to work even a little bit.  To be clear, I am unhappy with a lot of stuff I've done so far, but I want to focus on building the critical infrastructure rather than play whack-a-mole with bugs.  The only thing that bugs me is that the jupyter notebook couldn't finish the final calculation even after 2 minutes of waiting.  But I'll have to fix that later.  

I've searched for an ascii conversion module, and come up somewhat short.  How might I implement this myself?

I've already mentioned that I want to define a class.  This would be a numeric class, which would definitely inherit from the `int` datatype.  The `__init__()` function would be a little bit screwy because I'd want to accept many different input types as the argument of the constructor.  I think `__index__()` gets implemented automatically because I'm inheriting from `int` but I'd need to check that.  Then I'd obviously need a `__str__()` method in case anybody tries to print the thing.  

Actually I should probably define how I'd like the objects in this class to work.  let's think.

```
>>> message = Cipher("hello")
>>> print(message)
xxxhello
>>> hex(message)
0x78787868656c6c6f
>>> int(message+1)
120120120104101108108112
>>> type(message*2)
<blah blah blah type Cipher>
```

So we should have automatic padding, it should print as a string, it should be easily passed into `int()` and `hex()` and other conversion functions, and it should keep its type under math operations.  That means division will act like floor division.  How would we actually do the conversion?

We'd need to 1) go through 8 byte chunks and 2) have a lookup table.  The lookup table probably isn't even that hard to implement, its just a dictionary, practically speaking.  and it only has like 128 entries.  

So how do we actually store the thing?

I think the easiest way is to store it as a numpy array of integers.  We can choose a word length for the cipher, and make the actual data a numpy array with n columns or rows or whatever's convenient.  This makes multiplication dead easy, but it makes addition and exponentiation kinda screwy.  

Also need to decide whether to pad the front or the back.  

okay, this sounds really hard, no matter how I do it.  I've done some more googling and it looks like the `codecs` module provides a lot of the functionality I need.  let's try using that, and see how things go.  

## `codecs`

Just thinking through how this ought to go, the table to get from letters to numbers will be bigger than the table to get back.  Because I want to be able to encode null characters and newlines and such, but when I convert back, I never want to see them again.

I've got a wonderful example from James Lim on StackOverflow.  I'll copy his code and modify it here.

I'm getting super confused here.  

TEXT CODECS:

ENCODE TEXT TO BYTES

DECODE BYTES TO TEXT

so in James Lim's example, the numbers are actually the "text", while the letters are the "encoding".  Thats counterintuitive, but I can get with it I guess.  

In my case, I'll encode strings (including printable, 33 emojis, and escape sequences) into bytes, and decode bytes into strings (including printables and 33 emojis).  That means I'll need my encoding dictionary to be longer than the decoding dictionary.  

In [2]:
emoji_bytes = [num.to_bytes() for num in chain(range(32), [127])]
emoji_list = [
    "\U0001F5FF",                              # null
    "\U0001f92f", "\U0001f60d", "\U0001f911",  # tc 1-3
    "\U0001F636", "\U0001f62d", "\U0001F642",  # tc 4-6
    "\U0001f514",                              # bell
    "\U0001F353", "\U0001F36C", "\U0001F3B2",  # fe 0-2
    "\u23F3", "\U0001F48E", "\U0001F4A9",      # fe 3-5
    "\u270c", "\u270b", "\U0001f978",          # so, si, tc7
    "\u2600", "\u2601", "\u26a1", "\u2b50",    # dc 1-4
    "\U0001F641", "\U0001F634", "\U0001f9ca",  # tc 8-10
    "\U0001f383", "\U0001f384",                # cancel, end of medium
    "\U0001f389", "\U0001f37e",                # substitute, escape
    "\u2660", "\u2663", "\u2665", "\u2666",    # is 1-4
    "\U0001F480",                              # delete
]
print(emoji_list)

['🗿', '🤯', '😍', '🤑', '😶', '😭', '🙂', '🔔', '🍓', '🍬', '🎲', '⏳', '💎', '💩', '✌', '✋', '🥸', '☀', '☁', '⚡', '⭐', '🙁', '😴', '🧊', '🎃', '🎄', '🎉', '🍾', '♠', '♣', '♥', '♦', '💀']


In [3]:
# prepare map from string to bytecode
_encode_ascii_bytes = [num.to_bytes() for num in range(128)]
_encode_ascii = {str(byte.decode("ascii")): byte for byte in _encode_ascii_bytes}

_encode_emoji = {emoji_list[x]: emoji_bytes[x] for x in range(len(emoji_bytes))}

_encode_table = {**_encode_ascii, **_encode_emoji}

print(len(_encode_table))

161


In [4]:
# prepare map from bytecode to string
_decode_ascii_bytes = [num.to_bytes() for num in range(32, 127)]
_decode_ascii = {byte: str(byte.decode("ascii")) for byte in _decode_ascii_bytes}

_decode_emoji = {emoji_bytes[x]: emoji_list[x] for x in range(len(emoji_bytes))}

_decode_table = {**_decode_emoji, **_decode_ascii}

print(len(_decode_table))

128


In [5]:
fucking_idek = [int(x.hex(), base=16)for x in _decode_table]
print(fucking_idek)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 127, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126]


In [6]:
def custom_encode(text: str) -> tuple:
    # example encoder that converts ints to letters
    # see https://docs.python.org/3/library/codecs.html#codecs.Codec.encode
    return b''.join(_encode_table[x] for x in text), len(text)


def custom_decode(binary: bytes) -> tuple:
    # example decoder that converts letters to ints
    # see https://docs.python.org/3/library/codecs.html#codecs.Codec.decode
    return ''.join(_decode_table[x.to_bytes()] for x in binary), len(binary)


def custom_search_function(encoding_name):
    return codecs.CodecInfo(custom_encode, custom_decode, name="emojiascii")

codecs.register(custom_search_function)

In [7]:
message = codecs.decode(b"\n\txhello", encoding="emojiascii")
print(message)

🎲🍬xhello


In [8]:
for n in range(3):
    print(n.to_bytes())

b'\x00'
b'\x01'
b'\x02'


Cool, that sorta works! I need to fix the emojis a little bit, but in theory, the ASCII encoding/decoding works.

Okay thanks to Cayley's help, the emoji encoding works great now.  So we move onto  a full implementation.  I'm gonna copy this code into a .py file somehow.  

In [9]:
# stolen from socal_nerdtastic on reddit

def invert(dictionary: dict):
    for key, value in dictionary.copy().items():
        del dictionary[key]
        dictionary[value] = key

In [10]:
_encode_emoji1 = {
    "\U0001F5FF": b'\x00',                                                         # null
    "\U0001f92f": b'\x01', "\U0001f60d": b'\x02', "\U0001f911": b'\x03',           # tc 1-3
    "\U0001F636": b'\x04', "\U0001f62d": b'\x05', "\U0001F642": b'\x06',           # tc 4-6
    "\U0001f514": b'\x07',                                                         # bell
    "\U0001F353": b'\x08', "\U0001F36C": b'\x09', "\U0001F3B2": b'\x0a',           # fe 0-2
    "\u23F3": b'\x0b', "\U0001F48E": b'\x0c', "\U0001F4A9": b'\x0d',               # fe 3-5
    "\u270c": b'\x0e', "\u270b": b'\x0f', "\U0001f978": b'\x10',                   # so, si, tc7
    "\u2600": b'\x11', "\u2601": b'\x12', "\u26a1": b'\x13', "\u2b50": b'\x14',    # dc 1-4
    "\U0001F641": b'\x15', "\U0001F634": b'\x16', "\U0001f9ca": b'\x17',           # tc 8-10
    "\U0001f383": b'\x18', "\U0001f384": b'\x19',                                  # cancel, end of medium
    "\U0001f389": b'\x1a', "\U0001f37e": b'\x1b',                                  # substitute, escape
    "\u2660": b'\x1c', "\u2663": b'\x1d', "\u2665": b'\x1e', "\u2666": b'\x1f',    # is 1-4
    "\U0001F480": b'\x7f',                                                         # delete
}

In [11]:
invert(_encode_emoji1)

In [12]:
print(_encode_emoji1)

{b'\x00': '🗿', b'\x01': '🤯', b'\x02': '😍', b'\x03': '🤑', b'\x04': '😶', b'\x05': '😭', b'\x06': '🙂', b'\x07': '🔔', b'\x08': '🍓', b'\t': '🍬', b'\n': '🎲', b'\x0b': '⏳', b'\x0c': '💎', b'\r': '💩', b'\x0e': '✌', b'\x0f': '✋', b'\x10': '🥸', b'\x11': '☀', b'\x12': '☁', b'\x13': '⚡', b'\x14': '⭐', b'\x15': '🙁', b'\x16': '😴', b'\x17': '🧊', b'\x18': '🎃', b'\x19': '🎄', b'\x1a': '🎉', b'\x1b': '🍾', b'\x1c': '♠', b'\x1d': '♣', b'\x1e': '♥', b'\x1f': '♦', b'\x7f': '💀'}


fuck.

In [13]:
_encode_ascii5 = {num.to_bytes().decode("ascii"): num.to_bytes() for num in range(128)}


In [15]:
print(_encode_ascii5)


{'\x00': b'\x00', '\x01': b'\x01', '\x02': b'\x02', '\x03': b'\x03', '\x04': b'\x04', '\x05': b'\x05', '\x06': b'\x06', '\x07': b'\x07', '\x08': b'\x08', '\t': b'\t', '\n': b'\n', '\x0b': b'\x0b', '\x0c': b'\x0c', '\r': b'\r', '\x0e': b'\x0e', '\x0f': b'\x0f', '\x10': b'\x10', '\x11': b'\x11', '\x12': b'\x12', '\x13': b'\x13', '\x14': b'\x14', '\x15': b'\x15', '\x16': b'\x16', '\x17': b'\x17', '\x18': b'\x18', '\x19': b'\x19', '\x1a': b'\x1a', '\x1b': b'\x1b', '\x1c': b'\x1c', '\x1d': b'\x1d', '\x1e': b'\x1e', '\x1f': b'\x1f', ' ': b' ', '!': b'!', '"': b'"', '#': b'#', '$': b'$', '%': b'%', '&': b'&', "'": b"'", '(': b'(', ')': b')', '*': b'*', '+': b'+', ',': b',', '-': b'-', '.': b'.', '/': b'/', '0': b'0', '1': b'1', '2': b'2', '3': b'3', '4': b'4', '5': b'5', '6': b'6', '7': b'7', '8': b'8', '9': b'9', ':': b':', ';': b';', '<': b'<', '=': b'=', '>': b'>', '?': b'?', '@': b'@', 'A': b'A', 'B': b'B', 'C': b'C', 'D': b'D', 'E': b'E', 'F': b'F', 'G': b'G', 'H': b'H', 'I': b'I', 'J': 